# Lab 7A: Genre Classification on GTZAN


### AIML ZG527: Audio Analysis
**Authored by:** Seetha Parameswaran, BITS Pilani WILP
### Learning Objectives
- Extract MFCC features from FMA-small audio clips
- Train Random Forest baseline for genre classification
- Evaluate using accuracy and per-genre classification report
---
### Setup and Import Libraries

```
!pip install librosa scikit-learn pandas --break-system-packages
```

### Dataset: GTZAN 
10 genres, 1000 tracks, 30s clips, ~1.2GB

```
!kaggle datasets download -d andradaolteanu/gtzan-dataset-music-genre-classification
!unzip gtzan-dataset-music-genre-classification.zip
```

In [1]:
import os, numpy as np, librosa
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

AUDIO_DIR = '/home/seetha/PythonScriptsCourses/PythonScriptsCourses/Audio Analysis/Data/gtzan/Data/genres_original'  # update path

## Load audio files per genre folder

In [2]:
genres = sorted(os.listdir(AUDIO_DIR))
print(genres)

['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']


## Feature extraction: MFCCs (mean+std aggregation)

In [3]:
def extract_features(path, sr=22050, n_mfcc=13):
    y_audio, sr = librosa.load(path, sr=sr, duration=30)
    mfcc = librosa.feature.mfcc(y=y_audio, sr=sr, n_mfcc=n_mfcc)
    return np.concatenate([mfcc.mean(axis=1), mfcc.std(axis=1)])

In [4]:
X, y = [], []
for genre in genres:
    genre_dir = os.path.join(AUDIO_DIR, genre)
    for fname in os.listdir(genre_dir):
        if fname.endswith('.wav'):
            try:
                X.append(extract_features(os.path.join(genre_dir, fname)))
                y.append(genre)
            except Exception as e:
                print('skip', fname, e)
X = np.array(X)
print(X.shape, len(y))

/tmp/ipykernel_145950/1714814277.py:2: UserWarning: PySoundFile failed. Trying audioread instead.
  y_audio, sr = librosa.load(path, sr=sr, duration=30)
/home/seetha/PythonScriptsCourses/PythonScriptsCourses/lib/python3.8/site-packages/librosa/core/audio.py:183: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


skip jazz.00054.wav 
(999, 26) 999


## Train/test split + Random Forest baseline

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
clf = RandomForestClassifier(n_estimators=200, random_state=42)
clf.fit(X_train, y_train)
pred = clf.predict(X_test)
print('Accuracy:', accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

Accuracy: 0.655
              precision    recall  f1-score   support

       blues       0.79      0.75      0.77        20
   classical       0.84      0.80      0.82        20
     country       0.71      0.60      0.65        20
       disco       0.45      0.50      0.48        20
      hiphop       0.59      0.50      0.54        20
        jazz       0.60      0.75      0.67        20
       metal       0.74      0.85      0.79        20
         pop       0.70      0.80      0.74        20
      reggae       0.56      0.45      0.50        20
        rock       0.58      0.55      0.56        20

    accuracy                           0.66       200
   macro avg       0.66      0.66      0.65       200
weighted avg       0.66      0.66      0.65       200



## Summary

In this lab, you learned:

1. **Feature Extraction**: MFCCs aggregated (mean+std) per track
2. **Baseline Model**: Random Forest on hand-crafted features
3. **Evaluation**: Accuracy, per-genre precision/recall/F1

### Key Takeaways
- Traditional ML plateaus around 65-72% on genre tasks — deep learning (CNN/Transformer) needed for higher accuracy
- Genre labels are inherently noisy/subjective — misclassifications often make musical sense
- FMA-small mirrors GTZAN's role as a public genre benchmark